# Field-Level Two-Point Cross-Check: sft-wick vs sachsfield MC

We validate the perturbative two-point correlation of the **field itself** (not the time-integrated field) against direct nonlinear simulation on the sphere.

**Observable:** $\xi(\theta, \lambda_f) = \left\langle \xi_1(\hat{n}_1, \lambda_f)\, \xi_1(\hat{n}_2, \lambda_f) \right\rangle$ where $\cos\theta = \hat{n}_1 \cdot \hat{n}_2$.

**Key factorization:** Since $C_\ell^{\rm src}(\tau_1,\tau_2) = \frac{A}{\ell(\ell+1)} e^{-c(\tau_1-\tau_2)^2}$ separates, the dressed C propagator factorizes:
$$C(t_1,t_2;\theta) = S(\theta) \times C_{\rm corr}(t_1,t_2)$$
where $S(\theta) = \sum_\ell \frac{2\ell+1}{4\pi} \frac{A}{\ell(\ell+1)} P_\ell(\cos\theta)$.

**Tree-level:** With external times fixed at $\lambda_f$, the tree-level power spectrum is simply:
$$C_\ell^{\xi,(0)} = \frac{A}{\ell(\ell+1)} \, C_{\rm corr}(\lambda_f, \lambda_f)$$

**Perturbative structure:**
- Order 0 (tree): 1 diagram $C_{11}(x,y)$, depends on $\theta$ via $S(\theta)$
- Orders 1, 3: **vanish** (odd operator count)
- Order 2: 1-loop corrections (6 diagrams)
- Order 4: 2-loop corrections (64 diagrams)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import sys, os
from types import SimpleNamespace
from collections import defaultdict
from scipy.interpolate import interp1d, RectBivariateSpline
from scipy.integrate import cumulative_trapezoid
from scipy.special import eval_legendre
import healpy as hp

# sachsfield (add parent to path)
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import sachsfield as sf
from sachsfield.saddle import solve_singular_theta
from sachsfield.fluctuations import FluctuationSolver

# sft-wick
from sft_wick import (
    Field, Vertex, Action, compute_moment, reset_uid_counter,
    PropagatorModel, PropagatorCache, integrate_moment, integrate_diagrams,
)

%matplotlib inline
plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

## 1. Parameters

In [ ]:
# --- Physical parameters ---
A = 1e-12              # source amplitude (large enough for visible loop corrections)
sbar_1 = -1e-6         # constant background for field 1
c = 0.01               # Gaussian correlation: exp(-c*(lam'-lam'')^2), corr length ~10
lam_f = 3000.0         # evaluation endpoint
epsilon = 1e-8         # small-time cutoff (saddle solver + sft-wick propagators)
lam_start = epsilon    # MC start: no singularity in xi(0)=0, so start from epsilon

# Full-sky grid
nside = 32
lmax = 3 * nside - 1   # = 95
npix = hp.nside2npix(nside)  # = 12288

# --- F tensor ---
F_physical = np.zeros((3, 3, 3))
F_physical[0, 0, 0] = -0.5
F_physical[0, 1, 1] = -2.0
F_physical[0, 2, 2] = -2.0
F_physical[1, 0, 1] = -1.0
F_physical[2, 0, 2] = -1.0
F_code = -1j * F_physical   # MSR convention

# --- Noise kernel amplitude ---
ells = np.arange(1, lmax + 1)
ell_sum = np.sum((2 * ells + 1) / (4 * np.pi * ells * (ells + 1)))
S0 = A * ell_sum

# --- sft-wick field setup ---
phi = Field('phi', 'physical', n_components=3)
psi = Field('psi', 'response', n_components=3)
v = Vertex(fields=[psi, phi, phi], coupling='F')
action = Action(vertices=[v])

# --- Angular separations to probe ---
theta_deg = np.array([1.0, 5.0, 15.0, 45.0])
theta_rad = np.deg2rad(theta_deg)
cos_theta_arr = np.cos(theta_rad)

print(f'A = {A:.2e}, sbar_1 = {sbar_1:.2e}, c = {c}')
print(f'lam_f = {lam_f}, lam_start = {lam_start:.2e} (= epsilon)')
print(f'nside = {nside}, lmax = {lmax}, npix = {npix}')
print(f'S0 = {S0:.6e}')
print(f'Angular separations: {theta_deg} deg')

## 2. Saddle Point and Response Propagator

Solve $d\theta/dt = -\tfrac12 \theta^2 + \bar{s}_1$ with $\theta \sim 2/t$ at the origin, then build $\Phi(t) = \int_0^t \theta(\tau)\,d\tau$ and $R(t,t') = e^{\Phi(t')-\Phi(t)}$.

In [ ]:
# --- Solve saddle point ---
# The saddle is still needed: it provides M(lam) for the fluctuation ODE
# and Phi(t) for the response propagator R(t,t').
t_theta, theta_vals = solve_singular_theta(
    s=lambda t: sbar_1,
    t_max=lam_f,
    epsilon=epsilon,
    num_points=6000,
    method='Radau',
)
print(f'Saddle solved: {len(t_theta)} points in [{t_theta[0]:.2e}, {t_theta[-1]:.1f}]')
print(f'theta(eps) = {theta_vals[0]:.4f},  2/eps = {2/epsilon:.4f}')
print(f'theta(lam_f) = {theta_vals[-1]:.8f}')

# --- Build corrected Phi interpolator ---
t_trans = 5.0
Phi_raw = cumulative_trapezoid(theta_vals, t_theta, initial=0.0)
Phi_analytic_at_trans = 2.0 * np.log(t_trans / epsilon)
k_trans = np.searchsorted(t_theta, t_trans)
Phi_numerical_at_trans = Phi_raw[k_trans]
Phi_offset = Phi_analytic_at_trans - Phi_numerical_at_trans
Phi_corrected = Phi_raw + Phi_offset
Phi_corrected[:k_trans] = 2.0 * np.log(t_theta[:k_trans] / epsilon)
Phi_interp = interp1d(t_theta, Phi_corrected, kind='cubic', fill_value='extrapolate')

print(f'Phi correction: offset = {Phi_offset:.1f}')
print(f'Phi(lam_f) = {Phi_corrected[-1]:.4f}')

# --- Response propagator ---
def R_time(t, t_prime):
    """R(t, t') = exp(Phi(t') - Phi(t)) for t > t' > 0, else 0."""
    if t <= 0 or t_prime <= 0 or t <= t_prime:
        return 0.0
    if t < epsilon or t_prime < epsilon:
        return 0.0
    diff = float(Phi_interp(t_prime) - Phi_interp(t))
    if diff < -500:
        return 0.0
    return float(np.exp(diff))

# --- ThetaSaddleResult wrapper for FluctuationSolver ---
# Since lam_start = epsilon, we use the same saddle solution for both
# the sft-wick propagators and the MC FluctuationSolver.
class ThetaSaddleResult:
    def __init__(self, t, theta, n_fields=3):
        self.lam = t
        self.n_fields = n_fields
        self._theta_interp = interp1d(t, theta, kind='cubic', fill_value='extrapolate')

    def __call__(self, lam):
        lam_arr = np.asarray(lam, dtype=float)
        scalar = lam_arr.ndim == 0
        lam_arr = np.atleast_1d(lam_arr)
        chi = np.zeros((self.n_fields, len(lam_arr)))
        chi[0] = self._theta_interp(lam_arr)
        return chi[:, 0] if scalar else chi

    def M(self, lam):
        theta_val = float(self._theta_interp(lam))
        return np.diag([-theta_val] * self.n_fields)

theta_saddle = ThetaSaddleResult(t_theta, theta_vals)

print(f'R(10,5) = {R_time(10.0,5.0):.6e}')
print(f'R(1000,500) = {R_time(1000.0,500.0):.6e}')

## 3. Direction-Independent C Kernel

Build $C_{\rm corr}(t_1, t_2) = V \cdot G \cdot V^T$ via the matrix method, **without** the $S_0$ prefactor (that goes into $S(\theta)$).

In [ ]:
# === Build C_corr table via matrix method (no S0 prefactor) ===
t0_build = time.time()

M_quad = 1000
q = np.linspace(epsilon, lam_f, M_quad)
dq = q[1] - q[0]
Phi_q = Phi_interp(q)

# Gaussian kernel with correlation parameter c
G = np.exp(-c * np.subtract.outer(q, q)**2)  # (M, M)

# Output grid for C table
N_out = 400
ts = np.linspace(epsilon, lam_f, N_out)
Phi_ts = Phi_interp(ts)

# Weight matrix: V[k, m] = exp(Phi(q_m) - Phi(ts_k)) * dq  if q_m <= ts_k
mask_v = q[None, :] <= ts[:, None]  # (N, M)
exponent = np.where(mask_v, Phi_q[None, :] - Phi_ts[:, None], -1e3)
V = np.exp(exponent) * dq  # (N, M)
V[~mask_v] = 0.0

# C_corr: direction-independent kernel (NO S0 prefactor)
C_corr_table = V @ G @ V.T  # (N, N)

# Build spline
C_corr_spline = RectBivariateSpline(ts, ts, C_corr_table)

# Key quantity for field-level observable
C_corr_at_lf = float(C_corr_spline.ev(lam_f, lam_f))

print(f'C_corr table: {N_out}x{N_out} in {time.time()-t0_build:.1f}s')
print(f'C_corr(lam_f, lam_f) = {C_corr_at_lf:.6e}')
print(f'S0 * C_corr(lam_f, lam_f) = {S0 * C_corr_at_lf:.6e}')
print(f'Symmetric: max|C - C.T| = {np.max(np.abs(C_corr_table - C_corr_table.T)):.2e}')

## 4. Angular Weight $S(\theta)$ and Tree-Level $C_\ell^{\xi}$

For the **field at fixed $\lambda_f$**, the tree-level angular power spectrum is:
$$C_\ell^{\xi,(0)} = \frac{A}{\ell(\ell+1)} \, C_{\rm corr}(\lambda_f, \lambda_f)$$
No time integration needed — just a single evaluation of $C_{\rm corr}$.

In [ ]:
# --- Angular weight function S(cos_theta) ---
def S_angular(cos_th):
    """S(theta) = sum_ell (2ell+1)/(4pi) * A/(ell*(ell+1)) * P_ell(cos_theta)."""
    result = 0.0
    for ell in range(1, lmax + 1):
        result += (2 * ell + 1) / (4 * np.pi) * A / (ell * (ell + 1)) * eval_legendre(ell, cos_th)
    return result

# Verify S(0 deg) = S0
S_at_zero = S_angular(1.0)
print(f'S(0 deg) = {S_at_zero:.6e}')
print(f'S0       = {S0:.6e}')
print(f'Match: {abs(S_at_zero - S0) / S0:.2e} relative error')

# Compute S at target angular separations
S_vals = np.array([S_angular(ct) for ct in cos_theta_arr])
print(f'\nS(theta) / S0 at target angles:')
for i, th in enumerate(theta_deg):
    print(f'  {th:5.1f} deg: S/S0 = {S_vals[i]/S0:+.6f}')

# --- Tree-level C_ell for field at lam_f ---
Cl_xi_tree = np.zeros(lmax + 1)
for ell in range(1, lmax + 1):
    Cl_xi_tree[ell] = A / (ell * (ell + 1)) * C_corr_at_lf

print(f'\nC_corr(lam_f, lam_f) = {C_corr_at_lf:.6e}')

# Tree-level xi(theta) via Legendre sum
def xi_from_Cl(Cl, cos_th):
    """xi(theta) = sum_ell (2ell+1)/(4pi) * C_ell * P_ell(cos_theta)."""
    result = 0.0
    for ell in range(1, len(Cl)):
        result += (2 * ell + 1) / (4 * np.pi) * Cl[ell] * eval_legendre(ell, cos_th)
    return result

xi_tree_analytic = np.array([xi_from_Cl(Cl_xi_tree, ct) for ct in cos_theta_arr])

# Verify: xi_tree should equal S(theta) * C_corr(lam_f, lam_f)
xi_tree_direct = S_vals * C_corr_at_lf
print(f'\nTree-level xi(theta) verification:')
for i, th in enumerate(theta_deg):
    print(f'  {th:5.1f} deg: Legendre={xi_tree_analytic[i]:.6e}  '
          f'S*C_corr={xi_tree_direct[i]:.6e}  '
          f'relerr={abs(xi_tree_analytic[i]-xi_tree_direct[i])/(abs(xi_tree_direct[i])+1e-30):.2e}')

## 5. TwoPointCache and Custom Integration

The standard `PropagatorCache` assigns one direction to all spatial points. For the two-point function we need different sky directions at $x$ and $y$.

**Key change from integrated-field notebook:** External times $x, y$ are **fixed at $\lambda_f$**, not sampled by QMC. Only internal vertex times are integrated.

In [ ]:
class TwoPointCache:
    """Duck-typed PropagatorCache for direction-dependent two-point functions.
    
    Uses the factorization: C(n1,t1; n2,t2) = S(n1.n2) * C_corr(t1,t2) * I_N.
    R propagator is direction-independent (scalar when iso_R=True).
    """
    
    def __init__(self, R_time_func, c_corr_spline, S_angular_func, n_components=3):
        self.model = SimpleNamespace(
            iso_R=True, diag_C=True, n_components=n_components,
            R_time=R_time_func,
        )
        self._R_time = R_time_func
        self._c_corr_spline = c_corr_spline
        self._S_angular = S_angular_func
        self._s_cache = {}  # cache S(cos_theta) for repeated direction pairs
    
    def _get_S(self, n1, n2):
        """Compute S(cos_theta) with caching by direction pair."""
        key = (tuple(n1), tuple(n2))
        if key not in self._s_cache:
            cos_th = float(np.dot(np.asarray(n1), np.asarray(n2)))
            cos_th = np.clip(cos_th, -1.0, 1.0)
            self._s_cache[key] = self._S_angular(cos_th)
        return self._s_cache[key]
    
    def R_time(self, t_left, t_right):
        return self._R_time(t_left, t_right)
    
    def R_product(self, r_pairs, times):
        result = 1.0
        for sl, sr in r_pairs:
            result *= float(self.R_time(times[sl], times[sr]))
        return result
    
    def C_value(self, n1, t1, n2, t2):
        """C_{ab}(n1,t1; n2,t2) = S(n1.n2) * C_corr(t1,t2) * delta_{ab}."""
        N = self.model.n_components
        s_val = self._get_S(n1, n2)
        c_corr = float(self._c_corr_spline.ev(t1, t2))
        C_mat = np.zeros((N, N))
        for a in range(N):
            C_mat[a, a] = s_val * c_corr
        return C_mat
    
    def C_diagonal(self, n, t1, n_prime=None, t2=None):
        """Diagonal [C_{00}, C_{11}, ...] as 1D array."""
        if n_prime is None:
            n_prime = n
        if t2 is None:
            t2 = t1
        s_val = self._get_S(n, n_prime)
        c_corr = float(self._c_corr_spline.ev(t1, t2))
        return np.full(self.model.n_components, s_val * c_corr)


def make_two_point_directions(spatial, n1_vec, n2_vec):
    """Build directions dict mapping R-connected groups to sky directions.
    
    Convention: external point 'x' -> n1_vec, 'y' -> n2_vec.
    Internal vertices inherit direction from their R-chain connectivity.
    """
    directions = {}
    for group in spatial.direction_groups:
        any_point = next(iter(group))
        dir_var = spatial.direction_map[any_point]
        if 'x' in group:
            directions[dir_var] = tuple(n1_vec)
        elif 'y' in group:
            directions[dir_var] = tuple(n2_vec)
        else:
            # Internal-only group — default to n1 (for disconnected diagrams)
            directions[dir_var] = tuple(n1_vec)
    return directions


def integrate_two_point_qmc(ig, lambda_f, cache, directions,
                             t_min=0.0, n_samples=2**14, seed=None):
    """QMC integration with external times fixed at lambda_f.
    
    Unlike the integrated-field version, external points x,y are pinned
    at lambda_f. Only internal vertex times are sampled by QMC.
    """
    from scipy.stats import qmc
    
    spatial = ig.spatial
    int_vars_parents_first = list(reversed(spatial.time_integration_vars))
    ext_vars = list(spatial.external_points)
    n_int = len(int_vars_parents_first)
    
    # Fix external times at lambda_f
    ext_times = {v: lambda_f for v in ext_vars}
    
    # If no internal variables, just evaluate directly
    if n_int == 0:
        times = dict(ext_times)
        val = ig.evaluate(times, directions, cache)
        return (val.real, 0.0)
    
    # Build causal parent map for internal vars
    parent_map = defaultdict(list)
    for earlier, later in spatial.time_orderings:
        if earlier in int_vars_parents_first:
            parent_map[earlier].append(later)
    
    sampler = qmc.Sobol(d=n_int, seed=seed)
    u_samples = sampler.random(n_samples)
    values = np.empty(n_samples)
    
    for s in range(n_samples):
        u = u_samples[s]
        times = dict(ext_times)  # start with external times fixed
        jacobian = 1.0
        
        # Internal vars: bounded by causal parents
        for k, var in enumerate(int_vars_parents_first):
            parents = parent_map.get(var, [])
            if parents:
                hi = min(times[p] for p in parents if p in times)
            else:
                hi = lambda_f
            lo = t_min
            width = hi - lo
            if width <= 0:
                jacobian = 0.0
                times[var] = lo
            else:
                times[var] = lo + u[k] * width
                jacobian *= width
        
        if jacobian == 0.0:
            values[s] = 0.0
            continue
        
        result = ig.evaluate(times, directions, cache)
        values[s] = result.real * jacobian
    
    estimate = float(np.mean(values))
    n_batches = 8
    batch_size = n_samples // n_batches
    batch_means = np.array([
        np.mean(values[i*batch_size:(i+1)*batch_size])
        for i in range(n_batches)
    ])
    std_error = float(np.std(batch_means, ddof=1) / np.sqrt(n_batches))
    return (estimate, std_error)

print('TwoPointCache and custom QMC integration defined.')

## 6. sft-wick Diagram Computation

Observable: $\langle \phi_1(x)\, \phi_1(y) \rangle$ with perturbative expansion up to order 4. External times $x, y$ are fixed at $\lambda_f$.

In [ ]:
# --- Enumerate diagrams ---
perturbative_order = 4

reset_uid_counter()
obs = [phi('1', 'x'), phi('1', 'y')]

t0_enum = time.time()
result_2pt = compute_moment(
    obs, action, order=perturbative_order,
    response_phase=True, diag_R=True, diag_C=True, iso_R=True,
)
print(f'compute_moment(order={perturbative_order}): {time.time()-t0_enum:.1f}s')

# Print diagram count per order
for order in range(perturbative_order + 1):
    dts = result_2pt.diagram_terms(order)
    n_diag = len(dts) if dts else 0
    if n_diag > 0:
        dt0 = dts[0]
        print(f'Order {order}: {n_diag} diagram(s), '
              f'{len(dt0.integration_vars)} int vars, {len(dt0.propagators)} propagators')
    else:
        print(f'Order {order}: 0 diagram(s) (vanishes)')

In [ ]:
# --- Visualize diagrams (order 0 and 2 only; order 4 has 64 diagrams) ---
print('Order 0 diagrams:')
result_2pt.draw_diagrams(order=0)
plt.show()
print('\nOrder 2 diagrams:')
result_2pt.draw_diagrams(order=2)
plt.show()

In [ ]:
# --- Integrate diagrams at multiple angular separations ---
# External times x, y are FIXED at lam_f (field-level observable)
n1_vec = np.array([0.0, 0.0, 1.0])  # north pole

# Adaptive QMC samples: higher orders need fewer (corrections are small)
n_samples_by_order = {0: 2**16, 2: 2**14, 4: 2**12}

two_pt_cache = TwoPointCache(R_time, C_corr_spline, S_angular, n_components=3)

xi_wick = np.zeros(len(theta_rad))
xi_wick_tree = np.zeros(len(theta_rad))
xi_wick_breakdown = {}

t0_wick = time.time()
for i_th, th in enumerate(theta_rad):
    n2_vec = np.array([np.sin(th), 0.0, np.cos(th)])
    print(f'\n--- theta = {theta_deg[i_th]:.1f} deg ---')
    
    total = 0.0
    order_bd = {}
    for order in range(perturbative_order + 1):
        dts = result_2pt.diagram_terms(order)
        if not dts:
            continue
        ns = n_samples_by_order.get(order, 2**12)
        
        order_total = 0.0
        order_err_sq = 0.0
        for dt in dts:
            ig = dt.build_integrand({'F': F_code})
            dirs = make_two_point_directions(ig.spatial, n1_vec, n2_vec)
            val, err = integrate_two_point_qmc(
                ig, lam_f, two_pt_cache, dirs,
                t_min=epsilon, n_samples=ns,
            )
            order_total += val
            order_err_sq += err**2
        
        order_bd[order] = order_total
        total += order_total
        print(f'  Order {order}: {order_total:+.6e} +/- {np.sqrt(order_err_sq):.1e}  ({len(dts)} diagrams)')
        
        if order == 0:
            xi_wick_tree[i_th] = order_total
    
    xi_wick[i_th] = total
    xi_wick_breakdown[i_th] = order_bd

print(f'\nsft-wick integration done in {time.time()-t0_wick:.1f}s')

# Cross-check tree level vs analytic
print(f'\nTree-level verification (QMC vs analytic S*C_corr):')
for i_th, th in enumerate(theta_deg):
    qmc_val = xi_wick_tree[i_th]
    ana_val = xi_tree_direct[i_th]
    relerr = abs(qmc_val - ana_val) / (abs(ana_val) + 1e-30)
    print(f'  {th:5.1f} deg: QMC={qmc_val:.6e}  analytic={ana_val:.6e}  relerr={relerr:.2e}')

## 7. sachsfield Monte Carlo

Evolve the full nonlinear equation using `FluctuationSolver(linear_only=False)` and extract the **field at $\lambda_f$** (not the time-integrated field). Use `hp.anafast` for optimal $C_\ell$ estimation.

In [ ]:
# --- Source power spectrum ---
def cl_func(lam):
    cl = np.zeros(lmax + 1)
    ell = np.arange(1, lmax + 1)
    cl[1:] = A / (ell * (ell + 1))
    return cl, cl.copy(), cl.copy()

corr_func = lambda dlam: np.exp(-c * dlam**2)

# --- Lambda sampling ---
lam_samples = np.linspace(lam_start, lam_f, 200)
dlam_grid = lam_samples[1] - lam_samples[0]
n_lam = len(lam_samples)
print(f'{n_lam} lambda samples in [{lam_samples[0]:.2f}, {lam_samples[-1]:.1f}]')

# --- MC realization: returns field xi_1(n_hat) at lam_f ---
def run_realization_map(seed):
    """Run one realization, return xi_1(n_hat, lam_f) healpy map."""
    source = sf.FullSkySource(
        cl_func=cl_func, lam_samples=lam_samples, nside=nside,
        corr_func=corr_func, seed=seed, n_fields=3,
    )
    source.generate()
    delta_s = source.delta_s_interpolator()
    
    fl = FluctuationSolver(
        theta_saddle, delta_s, source.npix,
        n_fields=3, linear_only=False,
        lam_span=(lam_start, lam_f),
        method='RK45', rtol=1e-6, atol=1e-12,
    )
    result = fl.solve(t_eval=lam_samples)
    
    # Field 1 at lam_f (last time step), all pixels
    xi1_map = result.xi[-1, 0, :]  # (npix,)
    return xi1_map

# Quick test
t0 = time.time()
xi1_test = run_realization_map(seed=0)
print(f'Single realization: {time.time()-t0:.1f}s')
print(f'xi_1(lam_f): mean={xi1_test.mean():.4e}, std={xi1_test.std():.4e}')

In [ ]:
# --- Multi-seed MC with per-seed xi(theta) extraction ---
N_seeds = 30

# Precompute Legendre polynomials for target angles
P_ell_matrix = np.zeros((len(theta_rad), lmax + 1))  # (n_theta, lmax+1)
for i_th, ct in enumerate(cos_theta_arr):
    for ell in range(1, lmax + 1):
        P_ell_matrix[i_th, ell] = (2 * ell + 1) / (4 * np.pi) * eval_legendre(ell, ct)

t0 = time.time()
Cl_per_seed = np.zeros((N_seeds, lmax + 1))
xi_per_seed = np.zeros((N_seeds, len(theta_rad)))

for seed in range(N_seeds):
    xi1_map = run_realization_map(seed=seed)
    cl = hp.anafast(xi1_map, lmax=lmax)
    Cl_per_seed[seed] = cl
    
    # Per-seed xi(theta) via Legendre sum
    xi_per_seed[seed] = P_ell_matrix @ cl
    
    if seed % 10 == 0:
        print(f'seed={seed}: Cl[1]={cl[1]:.4e}, xi(5 deg)={xi_per_seed[seed, 1]:.4e}')

# Average C_ell and xi(theta) over seeds
Cl_mean = np.mean(Cl_per_seed, axis=0)
Cl_err = np.std(Cl_per_seed, axis=0) / np.sqrt(N_seeds)

xi_mc = np.mean(xi_per_seed, axis=0)
xi_mc_err = np.std(xi_per_seed, axis=0) / np.sqrt(N_seeds)

total_mc_time = time.time() - t0
print(f'\nMC complete: {N_seeds} seeds x {npix} pixels in {total_mc_time:.1f}s')
print(f'Cl[1] = {Cl_mean[1]:.4e} +/- {Cl_err[1]:.4e}')
print(f'Cl[10] = {Cl_mean[10]:.4e} +/- {Cl_err[10]:.4e}')

In [ ]:
# --- MC xi(theta) results ---
print('MC xi(theta) — averaged over', N_seeds, 'seeds:')
for i_th, th in enumerate(theta_deg):
    print(f'  {th:5.1f} deg: xi = {xi_mc[i_th]:.6e} +/- {xi_mc_err[i_th]:.6e}')

## 8. Comparison

Compare the two-point angular correlation of $\xi_1(\hat{n}, \lambda_f)$ from:
1. **sft-wick tree** (order 0 only)
2. **sft-wick full** (order 0 + 2 + 4)
3. **sachsfield MC** (nonlinear evolution, $N$ seeds)

In [ ]:
# --- Comparison table ---
print('=== Field-Level Two-Point Correlation: sft-wick vs MC ===')
print(f'{"theta":>8} | {"tree":>14} | {"O2 corr":>14} | {"O4 corr":>14} | {"wick total":>14} | {"MC":>14} | {"MC err":>12} | {"sigma":>6}')
print('-' * 110)
for i_th in range(len(theta_rad)):
    wt = xi_wick_tree[i_th]
    bd = xi_wick_breakdown[i_th]
    o2 = bd.get(2, 0.0)
    o4 = bd.get(4, 0.0)
    w = xi_wick[i_th]
    m = xi_mc[i_th]
    e = xi_mc_err[i_th]
    sig = abs(w - m) / e if e > 0 else 0
    print(f'{theta_deg[i_th]:>6.1f}\u00b0 | {wt:+14.6e} | {o2:+14.6e} | {o4:+14.6e} | '
          f'{w:+14.6e} | {m:+14.6e} | {e:12.6e} | {sig:6.1f}')

# Loop correction significance
print(f'\nLoop corrections relative to tree:')
for i_th in range(len(theta_rad)):
    wt = xi_wick_tree[i_th]
    bd = xi_wick_breakdown[i_th]
    o2_frac = bd.get(2, 0.0) / wt if wt != 0 else 0
    o4_frac = bd.get(4, 0.0) / wt if wt != 0 else 0
    print(f'  {theta_deg[i_th]:5.1f} deg: O2/tree = {o2_frac:+.6e}, O4/tree = {o4_frac:+.6e}')

In [ ]:
# --- Plots ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- C_ell comparison ---
ax = axes[0]
ells_plot = np.arange(1, lmax + 1)
ax.plot(ells_plot, Cl_xi_tree[1:], 'C0-', lw=2, label='sft-wick tree')
ax.errorbar(ells_plot, Cl_mean[1:], yerr=Cl_err[1:], fmt='C1.', ms=3,
            capsize=1, elinewidth=0.5, label='MC')
ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'$C_\ell^{\xi}$')
ax.set_title(r'Angular power spectrum of $\xi_1(\hat{n}, \lambda_f)$')
ax.legend()
ax.set_yscale('log')

# --- xi(theta) comparison ---
ax = axes[1]
ax.plot(theta_deg, xi_wick_tree, 'C0o--', lw=1.5, ms=7, label='sft-wick tree')
ax.plot(theta_deg, xi_wick, 'C2s-', lw=2, ms=7, label=f'sft-wick full (O{perturbative_order})')
ax.errorbar(theta_deg, xi_mc, yerr=xi_mc_err, fmt='C1D-', lw=2, ms=6,
            capsize=3, label='MC (nonlinear)')
ax.set_xlabel(r'$\theta$ [deg]')
ax.set_ylabel(r'$\xi(\theta)$')
ax.set_title(r'Two-point angular correlation of $\xi_1$ at $\lambda_f$')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- C_ell ratio plot (MC / tree) ---
fig, ax = plt.subplots(figsize=(7, 4.5))

ratio = Cl_mean[1:] / Cl_xi_tree[1:]
ratio_err = Cl_err[1:] / Cl_xi_tree[1:]
ax.errorbar(ells_plot, ratio, yerr=ratio_err, fmt='C1.', ms=3,
            capsize=1, elinewidth=0.5, label='MC / tree')
ax.axhline(1.0, color='C0', ls='--', lw=1.5, label='tree = 1')
ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'$C_\ell^{\rm MC} / C_\ell^{\rm tree}$')
ax.set_title(r'Power spectrum ratio: MC / sft-wick tree')
ax.legend()
ax.set_ylim(0.5, 1.5)
plt.tight_layout()
plt.show()

## 9. Summary

In [ ]:
print('='*70)
print('FIELD-LEVEL TWO-POINT CORRELATION CROSS-CHECK — SUMMARY')
print('='*70)
print(f'\nObservable: xi(theta) = <xi_1(n1, lam_f) xi_1(n2, lam_f)>')
print(f'  lam_f = {lam_f}, lam_start = {lam_start:.2e}, lmax = {lmax}')
print(f'  A = {A:.2e}, sbar_1 = {sbar_1:.2e}, c = {c}')
print(f'  C_corr(lam_f, lam_f) = {C_corr_at_lf:.6e}')
print(f'  Perturbative order = {perturbative_order}')
print(f'  MC seeds = {N_seeds}')

print(f'\n{"theta":>8} | {"wick total":>14} | {"MC":>14} | {"MC err":>12} | {"sigma":>6}')
print('-' * 65)
for i_th in range(len(theta_rad)):
    w = xi_wick[i_th]
    m = xi_mc[i_th]
    e = xi_mc_err[i_th]
    sig = abs(w - m) / e if e > 0 else 0
    print(f'{theta_deg[i_th]:>6.1f}\u00b0 | {w:+14.6e} | {m:+14.6e} | {e:12.6e} | {sig:6.1f}')

print('\n' + '='*70)